<a href="https://colab.research.google.com/github/rakhimahale13-collab/molecompare-qiskit/blob/main/notebooks/01_H2_VQE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoleCompare — H₂ Quantum Energy Experiment

This notebook estimates the ground-state energy of the hydrogen molecule (H₂) using the Variational Quantum Eigensolver (VQE).

**Goal:** Demonstrate a quantum-classical workflow for molecular energy estimation and compare the VQE result with a classical reference value.

In [1]:
!pip install -q qiskit qiskit-algorithms qiskit-nature qiskit-nature-pyscf pyscf matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

In [4]:
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.741",
    basis="sto3g",
)

problem = driver.run()

print("Number of spatial orbitals:", problem.num_spatial_orbitals)
print("Number of particles:", problem.num_particles)

Number of spatial orbitals: 2
Number of particles: (1, 1)


In [6]:
mapper = JordanWignerMapper()

initial_state = HartreeFock(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
)

ansatz = UCCSD(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
)

print("Number of qubits:", ansatz.num_qubits)
print("Number of trainable parameters:", ansatz.num_parameters)

Number of qubits: 4
Number of trainable parameters: 3


In [7]:
estimator = StatevectorEstimator()
optimizer = SLSQP(maxiter=100)

vqe_solver = VQE(estimator, ansatz, optimizer)
vqe_solver.initial_point = np.zeros(ansatz.num_parameters)

ground_state_solver = GroundStateEigensolver(mapper, vqe_solver)
result = ground_state_solver.solve(problem)

print(result)

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


=== GROUND STATE ENERGY ===
 
* Electronic ground state energy (Hartree): -1.851413691463
  - computed part:      -1.851413691463
~ Nuclear repulsion energy (Hartree): 0.714139285992
> Total ground state energy (Hartree): -1.137274405471
 
=== MEASURED OBSERVABLES ===
 
  0:  # Particles: 2.000 S: 0.000 S^2: 0.000 M: 0.000
 
=== DIPOLE MOMENTS ===
 
~ Nuclear dipole moment (a.u.): [0.0  0.0  1.40028706]
 
  0: 
  * Electronic dipole moment (a.u.): [0.0  0.0  1.400286879516]
    - computed part:      [0.0  0.0  1.400286879516]
  > Dipole moment (a.u.): [0.0  0.0  0.000000180484]  Total: 0.000000180484
                 (debye): [0.0  0.0  0.000000458744]  Total: 0.000000458744
 


In [8]:
vqe_energy = result.total_energies[0]

print("VQE total ground-state energy:", vqe_energy, "Hartree")

VQE total ground-state energy: -1.1372744054709973 Hartree


In [9]:
reference_energy = -1.13727
absolute_error = abs(vqe_energy - reference_energy)

print("Reference energy:", reference_energy, "Hartree")
print("VQE energy:", vqe_energy, "Hartree")
print("Absolute error:", absolute_error, "Hartree")
print("Chemical accuracy achieved:", absolute_error < 0.0016)

Reference energy: -1.13727 Hartree
VQE energy: -1.1372744054709973 Hartree
Absolute error: 4.405470997337346e-06 Hartree
Chemical accuracy achieved: True
